In [18]:
from datasets import load_dataset

from transformers import AutoTokenizer
from transformers import DataCollatorWithPadding
from transformers import AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer

## Load & Preprocess

In [19]:
dataset = load_dataset("csv", data_files="data/train.csv")["train"]

tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")

In [20]:
def preprocess_input(batch):
    """Unifies the prompts and responses into a single sentence and creates a single integer label from the label columns."""
    prompts = []
    for prompt, resp_a, resp_b in zip(batch["prompt"], batch["response_a"], batch["response_b"]):
        prompts.append(f"<prompt>{prompt}\n\n<answer1>{resp_a}\n\n<answer2>{resp_b}")

    labels = []
    for win_a, win_b in zip(batch["winner_model_a"], batch["winner_model_b"]):
        label = 0 if win_a else 2 if win_b else 1
        labels.append(label)
    return {**tokenizer(prompts, truncation=True), "labels": labels}

In [24]:
preprocessed = dataset.map(preprocess_input, batched=True)
dataset_split = preprocessed.train_test_split(test_size=0.1) # shuffled by default

In [10]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [11]:
id2label = {0: "Win A", 1: "Tie", 2: "Win B"}
label2id = {v:k for k, v in id2label.items()}

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert/distilbert-base-uncased", num_labels=3, id2label=id2label, label2id=label2id
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert/distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## Training

In [25]:
training_args = TrainingArguments(
    output_dir="out",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset_split["train"],
    eval_dataset=dataset_split["test"],
    processing_class=tokenizer,
    data_collator=data_collator
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss


## Inference

In [9]:
# the model gets loaded automatically after train
# only call this if no training was conducted
model = AutoModelForSequenceClassification.from_pretrained(
    "out/checkpoint-7186/"
)

In [ ]:
test_data = load_dataset("csv", data_files={"test": "data/test.csv"})["test"]
test_data = dataset.map(test_data, batched=True)

In [41]:
def combine(sample):
    """Combine the sample to the correct format."""
    return {"text": f"<prompt>{sample["prompt"]}\n\n<answer1>{sample["response_a"]}\n\n<answer2>{sample["response_b"]}"}

test_ds = dataset["test"].map(combine)

In [61]:
import torch
from collections import defaultdict

outs = defaultdict(list)

model = model.cpu()
for sample in test_ds:
    tokenized = tokenizer(sample["text"], return_tensors="pt", truncation=True)
    # tokenized = tokenized.to("cuda:0")
    with torch.no_grad():
        logits = model(**tokenized).logits
    probs = logits.softmax(dim=-1)[0].tolist()
    outs["id"].append(sample["id"])
    outs["winner_model_a"].append(probs[0])
    outs["winner_model_b"].append(probs[2])
    outs["tie"].append(probs[1])

defaultdict(list,
            {'id': [136060, 211333, 1233961],
             'winner_model_a': [0.23093360662460327,
              0.3105784058570862,
              0.35215452313423157],
             'winner_model_b': [0.22283044457435608,
              0.43917638063430786,
              0.37008363008499146],
             'tie': [0.546235978603363,
              0.25024521350860596,
              0.2777618169784546]})

In [63]:
import pandas as pd

outs = pd.DataFrame(outs)
outs.to_csv("submission.csv", index=False)

In [64]:
type(tokenizer)

transformers.models.distilbert.tokenization_distilbert_fast.DistilBertTokenizerFast